In [1]:
import numpy as np
from scipy.special import gammaln
import jupyter_black
jupyter_black.load()

In [2]:
def _log_ratio(k, log_lambda, y, sigma2):
    """
    log(a_{k+1} / a_k)
    """
    return log_lambda - np.log(k + 1.0) + (y - k - 0.5) / sigma2


def _find_mode(lam, y, sigma2):
    """
    Find k_star such that a_k is maximized.

    Uses the fact that log_ratio(k) is strictly decreasing.
    """
    if lam == 0.0:
        return 0

    log_lambda = np.log(lam)

    # If a_1 <= a_0, mode is k=0.
    if _log_ratio(0, log_lambda, y, sigma2) <= 0.0:
        return 0

    # Exponentially bracket the mode.
    lo = 0
    hi = 1

    while _log_ratio(hi, log_lambda, y, sigma2) > 0.0:
        lo = hi
        hi = 2 * hi + 1

    # Binary search for the first k with ratio <= 1.
    while lo + 1 < hi:
        mid = (lo + hi) // 2

        if _log_ratio(mid, log_lambda, y, sigma2) > 0.0:
            lo = mid
        else:
            hi = mid

    return hi


def _log_tail_bound(log_a, log_q):
    """
    Given a current term a and a geometric ratio q < 1,
    return

        log[a * q / (1-q)].

    If q >= 1, the geometric bound is unavailable.
    """
    if log_q >= 0.0:
        return np.inf

    q = np.exp(log_q)

    if q == 0.0:
        return -np.inf

    # log(q / (1-q)), written this way for numerical
    # stability when q is close to 1.
    return log_a + log_q - np.log(-np.expm1(log_q))


def log_poisson_gaussian_pdf(y, lam, sigma2, tol=1e-12):
    """
    Numerically stable Poisson-Gaussian log likelihood.

    Parameters
    ----------
    y : float
        Observed detector value.
    lam : float
        Poisson mean f(x; theta), must be >= 0.
    sigma2 : float
        Gaussian readout variance, must be > 0.
    tol : float
        Maximum relative truncation error.

    Returns
    -------
    logp : float
        Approximate log p(y | lam, sigma2).
    k_left : int
        Lower truncation boundary.
    k_right : int
        Upper truncation boundary.
    """

    if lam < 0.0:
        raise ValueError("Poisson rate lam must be non-negative.")

    if sigma2 <= 0.0:
        raise ValueError("sigma2 must be strictly positive.")

    # Special case: Poisson mean exactly zero.
    if lam == 0.0:
        log_norm = -0.5 * np.log(2.0 * np.pi * sigma2)
        logp = log_norm - y * y / (2.0 * sigma2)
        return logp, 0, 0

    log_lambda = np.log(lam)

    log_norm = -0.5 * np.log(2.0 * np.pi * sigma2)

    # ---------------------------------------------------------
    # 1. Find the mode
    # ---------------------------------------------------------
    k_mode = _find_mode(lam, y, sigma2)

    # log a_{k_mode}
    log_a_mode = (
        k_mode * log_lambda
        - lam
        - gammaln(k_mode + 1.0)
        + log_norm
        - (y - k_mode) ** 2 / (2.0 * sigma2)
    )

    # log of retained sum S
    log_S = log_a_mode

    # ---------------------------------------------------------
    # 2. Expand to the right
    # ---------------------------------------------------------
    k = k_mode
    log_a = log_a_mode

    while True:
        log_q = _log_ratio(k, log_lambda, y, sigma2)

        # Bound on all terms k+1, k+2, ...
        log_tail = _log_tail_bound(log_a, log_q)

        # Allocate half of tolerance to this tail.
        if log_tail <= log_S + np.log(tol / 2.0):
            break

        # Move one step to the right.
        k += 1
        log_a += log_q

        # Stable log-space accumulation:
        log_S = np.logaddexp(log_S, log_a)

    k_right = k

    # ---------------------------------------------------------
    # 3. Expand to the left
    # ---------------------------------------------------------
    k = k_mode
    log_a = log_a_mode

    while k > 0:
        # a_{k-1} / a_k
        log_q = -_log_ratio(k - 1, log_lambda, y, sigma2)

        # Bound on all terms k-1, k-2, ...
        log_tail = _log_tail_bound(log_a, log_q)

        if log_tail <= log_S + np.log(tol / 2.0):
            break

        # Move one step to the left.
        k -= 1
        log_a += log_q

        log_S = np.logaddexp(log_S, log_a)

    k_left = k

    return log_S, k_left, k_right

In [3]:
def poisson_gaussian_energy(y, f_values, sigma2, tol=1e-12):
    """
    Negative log likelihood for independent observations.
    """

    y = np.asarray(y, dtype=float)
    f_values = np.asarray(f_values, dtype=float)

    if y.shape != f_values.shape:
        raise ValueError("y and f_values must have the same shape.")

    energy = 0.0

    for yi, lam in zip(y, f_values):
        logp, _, _ = log_poisson_gaussian_pdf(yi, lam, sigma2, tol=tol)
        energy -= logp

    return energy